# 05. Bipolar и bridge PWM

Для signed-входа `[-1, 1]` в текущем коде есть два связанных подхода:

- `pwm_kind*_bipolar(...)`: две логические ветви `positive` и `negative`, полезный сигнал `positive - negative`.
- `pwm_kind*_bridge(...)`: физические выходы `plus` и `minus` для мостовой схемы, с режимами `regular`, `bipolar`, `three_level`.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if (HERE / "tutorial_helpers.py").exists():
    TUTORIAL_DIR = HERE
elif (HERE / "tutorials" / "tutorial_helpers.py").exists():
    TUTORIAL_DIR = HERE / "tutorials"
else:
    raise RuntimeError("Run this notebook from the repository root or from the tutorials folder")

path_text = str(TUTORIAL_DIR)
if path_text not in sys.path:
    sys.path.insert(0, path_text)

import matplotlib.pyplot as plt
import numpy as np

from tutorial_helpers import (
    configure_plots,
    grouped_fifo_channel_waveforms,
    load_pwm_lab,
    plot_bitstream,
    plot_channel_stack,
    plot_moving_average_reconstruction,
    plot_pwm_carrier_output,
    plot_spectra,
    print_peak_table,
    pwm_kind2_channel_waveforms,
    show_grouped_mapping,
    time_us,
)

pl = load_pwm_lab()
configure_plots()

## Signed FIFO-вход

In [ ]:
config = pl.PwmConfig(f_clk=80e6, f_pwm=1e6, resolution_bits=8)
f_signal = 50e3
n_periods = 128

_, x_signed = pl.sine_signed(freq=f_signal, sample_rate=config.actual_f_pwm, n_samples=n_periods, amplitude=0.85)

bipolar = pl.pwm_kind2_bipolar_latched(x_signed, config)
bridge_regular = pl.pwm_kind2_bridge_latched(x_signed, config, mode="regular")
bridge_bipolar = pl.pwm_kind2_bridge_latched(x_signed, config, mode="bipolar")
bridge_three = pl.pwm_kind2_bridge_latched(x_signed, config, mode="three_level")

## Bipolar positive/negative/differential

In [ ]:
n = 14 * config.period_samples
t = time_us(config.f_clk, n)
x_latched = np.repeat(x_signed, config.period_samples)

fig, axes = plt.subplots(4, 1, figsize=(11, 6.4), sharex=True, constrained_layout=True)
axes[0].plot(t, x_latched[:n], color="C1")
axes[0].set_title("Bipolar PWM kind 2")
axes[0].set_ylabel("input")
axes[1].step(t, bipolar.positive[:n], where="post", color="C0")
axes[1].set_ylabel("positive")
axes[2].step(t, bipolar.negative[:n], where="post", color="C3")
axes[2].set_ylabel("negative")
axes[3].step(t, bipolar.differential[:n], where="post", color="C2")
axes[3].set_ylabel("diff")
axes[3].set_xlabel("time, us");

## Bridge modes

`three_level` обычно ближе к split-magnitude логике: около нуля обе ветви выключены. `regular` всегда держит одну из физических ветвей активной.

In [ ]:
n = 14 * config.period_samples
t = time_us(config.f_clk, n)

fig, axes = plt.subplots(3, 1, figsize=(11, 6.0), sharex=True, constrained_layout=True)
for ax, obj, title in [
    (axes[0], bridge_regular, "regular differential"),
    (axes[1], bridge_bipolar, "bipolar differential"),
    (axes[2], bridge_three, "three-level differential"),
]:
    ax.step(t, obj.differential[:n], where="post", linewidth=1.0)
    ax.set_ylabel("diff")
    ax.set_title(title)
axes[2].set_xlabel("time, us");

## Спектры differential-выходов

In [ ]:
plot_spectra(
    {
        "bipolar helper": bipolar.differential,
        "bridge regular": bridge_regular.differential,
        "bridge bipolar": bridge_bipolar.differential,
        "bridge three-level": bridge_three.differential,
    },
    sample_rate=config.f_clk,
    f_max=10e6,
    f_scale=1e6,
    f_unit="MHz",
    title="Bipolar and bridge differential spectra",
);